### Voice Agents

In [13]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import Audio, display

In [2]:
load_dotenv()

client = OpenAI()

Verify with a Simple LLM Call

In [3]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{
        'role': 'user',
        'content': 'Say "Pipeline ready" in three words.'
    }]
)

print(response.choices[0].message.content)

Pipeline is ready.


Generate Audio with TTS
* Use OpenAI TTS to generate a small audio file

In [5]:
sample_text = 'How do I treat cassava mosaic disease?'

# Path where the audio file will be saved
audio_path = Path(r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3')

# Is data folder exists
audio_path.parent.mkdir(parents=True, exist_ok=True)

# Call OpenAI's TTS API
response = client.audio.speech.create(
    model='tts-1',
    voice='alloy',
    input=sample_text,
)

# Write the bytes to the file
audio_path.write_bytes(response.content)

# Save the audio bytes to a file
print(f'Audio saved: {audio_path}')
print(f'Size: {audio_path.stat().st_size / 1024:.1f} KB')

Audio saved: C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3
Size: 37.1 KB


Verify

In [7]:
display(Audio(str(audio_path)))

Step 4: Speech-to-Text (STT)

In [8]:
# Path to the audio file 
audio_path = Path(r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3')

# Open the file and send it to Whisper for transcription
with open(audio_path, 'rb') as f:
    transcript = client.audio.transcriptions.create(
        model='whisper-1',
        file=f,
    )
    
print(f'Transcribed text: {transcript.text}')

Transcribed text: How do I treat cassava mosaic disease?


Step 5a: Full Voice Pipeline

In [9]:
def voice_pipeline(audio_input_path: str, output_path: str) -> str:
    '''
    Full voice pipeline: audio in → text → LLM → text → audio out.
    '''
    
    # Layer 1: STT
    with open(audio_input_path, 'rb') as f:
        transcript = client.audio.transcriptions.create(
            model='whisper-1',
            file=f,
        )
        
    user_text = transcript.text
    print(f'📝 STT: {user_text}')
    
    # Layer 2: LLM 
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                'role': 'system',
                'content': 'You are a helpful farming assistant. Answer briefly.'
            },
            {
                'role': 'user',
                'content': user_text
            }
        ],
    )
    
    answer_text = response.choices[0].message.content
    print(f'💬 LLM: {answer_text}')
    
    # Layer 3: TTS
    tts_response = client.audio.speech.create(
        model='tts-1',
        voice='alloy',
        input=answer_text
    )
    Path(output_path).write_bytes(tts_response.content)
    print(f'🔊 TTS: saved to {output_path}')
    
    return answer_text

Step 5b: Test the Full Pipeline

In [10]:
input_audio  = r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3'
output_audio = r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_answer.mp3'

answer = voice_pipeline(input_audio, output_audio)
print(f'\n📋 Final answer: {answer}')

📝 STT: How do I treat cassava mosaic disease?
💬 LLM: To treat cassava mosaic disease, follow these management strategies:

1. **Use Resistant Varieties**: Plant disease-resistant cassava varieties.
2. **Rogue Infected Plants**: Regularly inspect and remove infected plants to reduce the spread.
3. **Control Insect Vectors**: Use insecticides or natural predators to manage cassava whiteflies, which spread the virus.
4. **Crop Rotation**: Practice crop rotation with non-host crops to break the disease cycle.
5. **Proper Farm Hygiene**: Clean tools and equipment to avoid cross-contamination.

Always consult local agricultural experts for specific practices suitable for your region.
🔊 TTS: saved to C:\Users\USER\rag_course\12b_voice_agents\data\sample_answer.mp3

📋 Final answer: To treat cassava mosaic disease, follow these management strategies:

1. **Use Resistant Varieties**: Plant disease-resistant cassava varieties.
2. **Rogue Infected Plants**: Regularly inspect and remove infected pl

Step 5c: Listen to the Answer

In [11]:
display(Audio(output_audio))

Step 6: Build the RAG-Powered Voice Pipeline

In [14]:
sys.path.append(r'C:\Users\USER\rag_course\13_agentic_rag\code\notebooks')

rag_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

embeddings = OpenAIEmbeddings()

Step 6b: Load the Vector Store

In [15]:
# Load the domain vector store
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs= {'k': 4})
print(f'Vector store loaded — {vectorstore._collection.count()} chunks')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store loaded — 80 chunks


Step 6c: Build the RAG Chain

In [18]:
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer using ONLY the context below. Keep it under 3 sentences for voice. If not found, say "I don\'t know".'),
    ('human', 'Context:\n{context}\n\nQuery: {query}'),
])

rag_chain = rag_prompt | rag_llm | StrOutputParser()

def rag_answer(question: str) -> str:
    '''
    Retrieve and answer in one call.
    '''
    
    docs = retriever.invoke(question)
    context = '\n\n'.join(d.page_content for d in docs)
    
    return rag_chain.invoke({
        'context': context,
        'query': question
    })
    
print('RAG chain reay')

RAG chain reay


Step 6d: The RAG-Powered Voice Pipeline

In [19]:
def voice_rag_pipeline(audio_input_path: str, output_path: str) -> str:
    '''
    Full voice pipeline: audio in -> STT -> RAG -> TTS -> audio out.
    '''
    
    # STT
    with open(audio_input_path, 'rb') as f:
        transcript = client.audio.transcriptions.create(
            model='whisper-1',
            file=f
        )
        
    user_text = transcript.text
    print(f'📝 STT: {user_text}')
    
    # RAG 
    answer_text = rag_answer(user_text)
    print(f'🧠 RAG: {answer_text}')
    
    # TTS
    tts_response = client.audio.speech.create(
        model='tts-1',
        voice='alloy',
        input=answer_text
    )
    Path(output_path).write_bytes(tts_response.content)
    print(f'🔊 TTS: saved to {output_path}')
    
    return answer_text

print('voice_rag_pipeline ready')

voice_rag_pipeline ready


Step 6e: Test It

In [20]:
input_audio  = r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3'
output_audio = r'C:\Users\USER\rag_course\12b_voice_agents\data\rag_answer.mp3'

answer = voice_rag_pipeline(input_audio, output_audio)
print(f'\n📋 Final answer: {answer}')

📝 STT: How do I treat cassava mosaic disease?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🧠 RAG: To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants early.
🔊 TTS: saved to C:\Users\USER\rag_course\12b_voice_agents\data\rag_answer.mp3

📋 Final answer: To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants early.
